In [ ]:
## Imports ##

# System Path #
import os
import sys 

# Add dsci_550_a1 to base path. Lets you project functions #
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

# Misc Data Handling #
import pandas as pd
import time
import re
import json 
from datetime import date 



# Runtime #
import time
from tqdm import tqdm 

# Iterators #
import collections
from itertools import chain
import ast
import random

# Flight Trajectory Functions #
from dsci_550_a1.flightFunctions import *

# Plotly #
import plotly.graph_objects as go
import plotly.express as px
import textwrap


In [200]:
# Helper Functions #
from dsci_550_a3.dg_viz import hp_interactive_globe
from dsci_550_a3.dg_query import filter_hp_df, get_legend_items
from dsci_550_a3.dg_dataLoader import load_all_data

# Pandas
import pandas as pd

# Parsing Date
from datetime import date
import ast
import json

# Preparing Haunted Place Images
import os
import base64


def prepare_hover_image(encoded_image):
    if pd.isna(encoded_image) or encoded_image == "":
        return "<b>No image available</b>"
    else:
        return f"<img src='{encoded_image}' width='120' height='80'>"

# Point to image
def encode_image(image_pointer, img_directory):

    # If there is no pointer, use default image
    if pd.isna(image_pointer) or image_pointer == "":
        img_path = os.path.join(img_directory, 'hpimg_placeholder.png')
    else:
        img_path = os.path.join(img_directory, 'hpimg_placeholder.png')
        #img_path = os.path.join(img_directory, image_pointer)
    
    # if file does not exist, use default
    if not os.path.exists(img_path):
        img_path = os.path.join(img_directory, 'hpimg_placeholder.png')
    
    # open file and encode
    with open(img_path, 'rb') as f:
        encoded = base64.b64encode(f.read()).decode()

    return 'data:image/png;base64,{}'.format(encoded)


# Encode image date for hover info
# def encode_image(image_pointer, img_directory):
#     try:
#         with open(image_pointer, 'rb') as f:
#             encoded = base64.b64encode(f.read()).decode()
#     except FileNotFoundError:
#         with open(prepare_hover_image('hpimg_placeholder.png', img_directory), 'rb') as f:
#             encoded = base64.b64encode(f.read()).decode()
#     return 'data:image/png;base64,{}'.format(encoded)

# Date parsing helper
def parse_date(s): 
    return date(*map(int, s.split('-'))) 

# Load Data
def load_all_data(img_directory):
    hp_df = pd.read_csv("../data/processed/haunted_places_features_added_v2.tab", sep="\t")
    hp_df['Haunted_Places_Date'] = hp_df['Haunted_Places_Date'].apply(lambda x: ast.literal_eval(x)) 
    hp_df['Haunted_Places_Date'] = hp_df['Haunted_Places_Date'].apply(lambda x: [parse_date(y) for y in x] if isinstance(x, list) else x)
    
    # Encode hpimg data
    hp_df['Image_Pointer'] = hp_df['Image_Pointer'].fillna('hpimg_placeholder.png')
    hp_df['Image_Pointer'] = hp_df['Image_Pointer'].apply(lambda x: encode_image(x, img_directory))

    route_df = pd.read_csv("../data/joined_datasets/american_routes.tsv", sep="\t")
    route_df["Flight_Path"] = route_df["Flight_Path"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    airport_df = pd.read_csv("../data/joined_datasets/american_airports.tsv", sep="\t")
    airport_df["Airport_Radius"] = airport_df["Airport_Radius"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    with open("../data/processed/flight_proximity_data.json", "r") as f:
        flight_intersections = json.load(f)

    with open("../data/processed/airport_proximity_data.json", "r") as f:
        airport_intersections = json.load(f)

    return (
        hp_df,
        route_df,
        airport_df,
        flight_intersections,
        airport_intersections
    )

hpimg_dir = "../data/generated_images"

## Load Data and Define Holidays
(
    hp_df,
    route_df,
    airport_df,
    flight_intersections,
    airport_intersections
) = load_all_data(hpimg_dir)

airport_types = ['heliport']
filtered_airport_df = airport_df[airport_df['Type'].isin(airport_types)].copy()

filtered_airport_df.head()

,Id,Ident,Type,Name,Latitude_Deg,Longitude_Deg,Elevation_Ft,Iso_Country,Iso_Region,Municipality,Scheduled_Service,Icao_Code,Iata_Code,Gps_Code,Local_Code,Airport_Radius
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,US,US-PA,Bensalem,no,0,0,K00A,00A,"[(40.09594019859284, -74.933689), (40.09010179..."
9,322658,00CN,heliport,Kitchen Creek Helibase Heliport,32.727374,-116.459742,3350.0,US,US-CA,Pine Valley,no,0,0,00CN,00CN,"[(32.75232879859284, -116.45974169999998), (32..."
13,6535,00GE,heliport,Caffrey Heliport,33.887982,-84.736983,957.0,US,US-GA,Hiram,no,0,0,00GE,00GE,"[(33.91293719859284, -84.736983), (33.90709879..."
14,6536,00HI,heliport,Kaupulehu Heliport,19.832881,-155.978347,43.0,US,US-HI,Kailua-Kona,no,0,0,00HI,00HI,"[(19.85783619859284, -155.978347), (19.8519977..."
18,6540,00IN,heliport,St Mary Medical Center Heliport,41.511398,-87.260597,634.0,US,US-IN,Hobart,no,0,0,00IN,00IN,"[(41.53635351402253, -87.2605972290039), (41.5..."


In [ ]:
from itertools import chain
from datetime import date
import regex as re

# Unique Legend Items
def get_legend_items(df_hp, legend_key):
    
    # return True and False if Bool
    if df_hp[legend_key].dtype == 'bool':
        return ['True', 'False']
    
    s = set().union(*df_hp[legend_key].dropna().str.split(' | ').tolist())
    
    try:
        s.remove('|')  # remove delimiter if it was caught
    
    except:
        pass
    return  sorted(list(s))

# String to Datetime
def parse_date(s): 
    return date(*map(int, s.split('-'))) 
# Datetime to String
def convert_date_str(date):
    return date.strftime('%Y-%m-%d')
# Date Range Filter
def in_date_range(date_list, start_date, end_date):
    return any(start_date <= date <= end_date for date in date_list)

def transform_year(year):
    '''
    Custom hard-coded mapping of years for year-slider. Necessary to fit all years on one slider.
    '''
    # 1600
    if year == 1780:
        return 1600
    # 1650
    elif year == 1785:
        return 1650
    # 1700
    elif year == 1790:
        return 1700
    # 1750
    elif year == 1795:
        return 1750
    return year

def query_df(query_keys, s):

    # Conver to list if single string is passed
    if isinstance(query_keys, str):
        query_keys = [query_keys]

    # Return False if df is null 
    if s is None:
        return False
    # Make logical "or" regex and query
    query_regex = "|".join(map(re.escape, query_keys))
    return bool(re.search(query_regex, s))

def filter_hp_df(
    hp_df,
    state=None, event_type=None, apparition_type=None, haunt_date_range=None, holiday = None):
    
    filtered_hp_df = hp_df.copy()

    if state:
        filtered_hp_df = filtered_hp_df[filtered_hp_df['State'].apply(lambda s: query_df(state, s))]
    if event_type:
        filtered_hp_df = filtered_hp_df[filtered_hp_df['Event_Type'].apply(lambda s: query_df(event_type, s))]
    if apparition_type:
        filtered_hp_df = filtered_hp_df[filtered_hp_df['Apparition_Type'].apply(lambda s: query_df(apparition_type, s))]
    if haunt_date_range:
        start_date, end_date = map(parse_date, haunt_date_range)
        filtered_hp_df = filtered_hp_df[(filtered_hp_df['Haunted_Places_Date'].apply(lambda x: in_date_range(x, start_date, end_date)))]
    if holiday:
        holiday = list(map(parse_date,holiday))
        filtered_hp_df = filtered_hp_df[filtered_hp_df['Haunted_Places_Date'].apply(lambda x: any([in_date_range(x, h, h) for h in holiday]))]
    
    filtered_hp_df['Haunted_Places_Date'] = filtered_hp_df['Haunted_Places_Date'].apply(lambda x: [convert_date_str(y) for y in x])
    filtered_hp_df = filtered_hp_df.astype(str)

    return filtered_hp_df

def filter_airport_df(
    filtered_hp_df,
    airport_df,
    flight_intersections,
    airport_intersections,
    airport_types : list = None):

    # if airport_types not specified, use all types
    if not airport_types:
        airport_types = airport_df['Type'].unique().tolist()
    
    
    filtered_airport_df = airport_df[airport_df['Type'].isin(airport_types)].copy()

    haunted_ids = filtered_hp_df['Haunted_Places_Id'].tolist()

    filtered_flight_intersections = {k: v['Routes'] for k, v in flight_intersections.items() if k in haunted_ids}
    filtered_airport_intersections = {k: v['Airports'] for k, v in airport_intersections.items() if k in haunted_ids}

    relevant_iata_codes = set()
    relevant_airports = set()

    for _, v in filtered_flight_intersections.items():
        relevant_iata_codes.update(
            chain(
            (route['Dest_Airport'] for route in v),
            (route['Source_Airport'] for route in v)
            )
        )
        
    for _, v in filtered_airport_intersections.items():
        relevant_airports.update(airport['Airport_ID'] for airport in v)

    filtered_airport_df = filtered_airport_df[filtered_airport_df['Id'].isin(relevant_airports) | filtered_airport_df['Iata_Code'].isin(relevant_iata_codes)]
    
    return filtered_airport_df

def filter_route_df(
    filtered_hp_df, 
    route_df, 
    flight_intersections):

    haunted_ids = filtered_hp_df['Haunted_Places_Id'].tolist()

    filtered_flight_intersections = {k: v['Routes'] for k, v in flight_intersections.items() if k in haunted_ids}

    relevant_routes = set()

    for _, v in filtered_flight_intersections.items():
        relevant_routes.update(route['Route_ID'] for route in v)

    filtered_route_df = route_df.loc[list(relevant_routes)]

    return filtered_route_df


legend_arg = None
state = None
event_type = None
apparition_type = None
haunt_date_range = ['1950-1-1', '2000-1-1']

#holiday = ['1000-10-31', '1000-12-24']


if not legend_arg:
    legend_arg = {'Event_Type': get_legend_items(hp_df, 'Event_Type')} 

filtered_hp_df = filter_hp_df(
    hp_df,
    state,
    event_type,
    apparition_type,
    haunt_date_range,
    holiday,
)
filtered_route_df = filter_route_df(filtered_hp_df, route_df, flight_intersections)
airport_types = ['large_airport']
filtered_airport_df = filter_airport_df(filtered_hp_df, airport_df, flight_intersections, airport_intersections, airport_types)

list(map(parse_date, haunt_date_range + holiday))


[datetime.date(1950, 1, 1), datetime.date(2000, 1, 1)]

In [ ]:
def generate_formatted_textbox(row, primary_tag, additional_tags, separator="|"):
    """
    Generate a formatted HTML string for a row based on specified fields.

    Parameters:
    - row: A pandas Series (a single row of a DataFrame).
    - fields: List of fields (column names) to include.
    - bold_fields: List of fields to bold (optional; if None, all fields will be bolded).
    - separator: String separator between lines (default: "<br>").
    - extra_formatting: Optional dictionary {field: custom_format_string}, where
                        custom_format_string can use {value} as placeholder.

    Returns:
    - A formatted HTML string.
    """
    lines = []
    lines.append(f"<b>Location</b>: {row['Location']} | <b>{primary_tag}</b>: {row.get(primary_tag, "")}")

    additional_tag_line = ['<b>Additional Tags:</b>'] + [f"{row.get(tag, '')}" for tag in additional_tags]
    lines.append(f" {separator} ".join(additional_tag_line))

    lines.append(f"<b>Number of Intersecting Flights</b>: {row['Flight_Intersection_Count']} | <b>Number of Nearby Airports</b>: {row['Aerodrome_Count']}<br>")
    lines.append(f"<b>Description</b>: {row['Formatted_Description']}")

    return "<br>".join(lines)





In [297]:
filtered_airport_df.head()

,Id,Ident,Type,Name,Latitude_Deg,Longitude_Deg,Elevation_Ft,Iso_Country,Iso_Region,Municipality,Scheduled_Service,Icao_Code,Iata_Code,Gps_Code,Local_Code,Airport_Radius
51,6572,00WI,small_airport,Northern Lite Airport,44.304298,-89.050102,860.0,US,US-WI,Waupaca,no,0,0,00WI,00WI,"[(44.35420879806459, -89.05010223388672), (44...."
811,7349,0MS0,small_airport,Topton Air Estates Airport,32.474998,-88.616699,453.0,US,US-MS,Meridian,no,0,0,0MS0,0MS0,"[(32.52490887130678, -88.61669921875), (32.513..."
1221,7810,13AZ,small_airport,Waltenberry Field Ultralightport,33.535000,-112.853472,1213.0,US,US-AZ,Tonopah,no,0,0,13AZ,13AZ,"[(33.58491039718567, -112.853472222), (33.5732..."
1262,7855,13TE,small_airport,Varisco Airport,30.656000,-96.538300,240.0,US,US-TX,Bryan,no,0,0,13TE,13TE,"[(30.705910534514786, -96.53829956054688), (30..."
1537,8135,19GA,small_airport,Willow Pond Aviation Inc Airport,33.424039,-84.498367,868.0,US,US-GA,Fayetteville,no,0,0,19GA,19GA,"[(33.47394939718568, -84.498367), (33.46227258..."


In [314]:
## Imports ##

# System Path #
import os
import sys 

# Misc Data Handling #
import pandas as pd
from datetime import date 

# Flight Trajectory Functions #
from dsci_550_a1.flightFunctions import *

# Plotly #
import plotly.graph_objects as go
import plotly.express as px
import textwrap


def prepare_hover_image(encoded_image):
    if pd.isna(encoded_image) or encoded_image == "":
        return "<b>No image available</b>"
    else:
        return f"<img src='{encoded_image}' width='120' height='80'>"

# Scaling size of haunted place markers
def scale_size(val, mn=5, mx=25):
    if mx == mn:
        return (mn + mx) / 2  
    return mn + (val - mn) * (mx - mn) / (mx - mn)


def airport_textbox(row):
    """
    Generate a formatted HTML string for a row based on specified fields.

    Parameters:
    - row: A pandas Series (a single row of a DataFrame).
    - fields: List of fields (column names) to include.
    - bold_fields: List of fields to bold (optional; if None, all fields will be bolded).
    - separator: String separator between lines (default: "<br>").
    - extra_formatting: Optional dictionary {field: custom_format_string}, where
                        custom_format_string can use {value} as placeholder.

    Returns:
    - A formatted HTML string.
    """
    lines = []
    name, iata, type = row.get('Name', ''), row.get('Iata_Code', ''), row.get("Type", "")
    # description w/o iata code
    if iata == '0' or iata == "":
        lines.append(f"<b>Name</b>: {name} | <b>Airport Type</b>: {type}")
    else:
    # description w/ iata code
        lines.append(f"<b>Name</b>: {name} ({iata}) | <b>Airport Type</b>: {type}")
    
    return "<br>".join(lines)


## Main Visualization Function ##
def hp_interactive_darkmatter(hp_df, route_df, airport_df, legend_arg, additional_tags = None, airport_visibility = []):
    # Initialize trace lists
    all_traces = []

    legend_key = list(legend_arg.keys())[0]
    legend_values = [v for v in legend_arg[legend_key]]

    # Color Palette
    plot_colors = {
    'dsci550_a3-1-hsla': 'rgb(116, 84, 190)', 'dsci550_a3-2-hsla': 'rgb(125, 133, 241)', 'dsci550_a3-3-hsla': 'rgb(58, 45, 113)', 'dsci550_a3-4-hsla': 'rgb(10, 188, 4)', 'dsci550_a3-5-hsla': 'rgb(77, 114, 23)', 
    'dsci550_a3-6-hsla': 'rgb(140, 215, 64)', 'dsci550_a3-7-hsla': 'rgb(237, 7, 7)', 'dsci550_a3-8-hsla': 'rgb(113, 3, 3)', 'dsci550_a3-9-hsla': 'rgb(188, 53, 4)',  'dsci550_a3-10-hsla': 'rgb(238, 114, 6)', 
    'dsci550_a3-11-hsla': 'rgb(241, 166, 74)', 'dsci550_a3-12-hsla': 'rgb(255, 137, 254)', 'dsci550_a3-13-hsla': 'rgb(116, 235, 148)', 'dsci550_a3-14-hsla': 'rgb(4, 214, 176)', 'dsci550_a3-15-hsla': 'rgb(65, 173, 240)', 
    'dsci550_a3-16-hsla': 'rgb(2, 100, 109)', 'dsci550_a3-17-hsla': 'rgb(1, 38, 59)', 'dsci550_a3-18-hsla': 'rgb(191, 176, 88)', 'dsci550_a3-19-hsla': 'rgb(34, 3, 1)', 'dsci550_a3-20-hsla': 'rgb(12, 12, 12)'
    }
        

    ## Haunted Places Trace 

    # Scaling for scatterplot diameters
    size_argumnet = 'Flight_Intersection_Count'
    mn_scale, mx_scale = hp_df[size_argumnet].astype(int).min(), hp_df[size_argumnet].astype(int).max()
    hp_df['Scaled_Size'] = hp_df[size_argumnet].astype(int).apply(scale_size, mn=mn_scale, mx=mx_scale)
    
    # Iterate through legend values
    for i, val in enumerate(legend_values):
        
        # Filter Dataset
        hp_df_filtered = hp_df.loc[hp_df[f'{legend_key}'].str.contains(val, na=False)].copy()
        hp_df_filtered['Formatted_Description'] = hp_df_filtered['Description'].apply(
        lambda x: "<br>".join(textwrap.wrap(x, width=50))
    )
        # Add Trace
        trace = (go.Scattermap(
            lon = hp_df_filtered['Longitude'],
            lat = hp_df_filtered['Latitude'],
            hoverinfo = 'skip',
            customdata = np.stack([
                                hp_df_filtered.apply(lambda row: generate_formatted_textbox(
                                                row,
                                                primary_tag=f"{legend_key}",
                                                additional_tags=additional_tags,
                                            ), 
                                        axis=1),
                                ], axis = 1
            ),
            hovertemplate = (
                "%{customdata[0]}<br><br>"
            ),
            mode = 'markers',
            showlegend = True, 
            marker = dict(
                size = hp_df_filtered['Scaled_Size'],
                sizemode = 'area',
                sizemin = 2,
                color = plot_colors[f"dsci550_a3-{i+1}-hsla"],
                opacity = 0.75
                ),
                name = val, 
                visible = True
            )
        )
        all_traces.append(trace)


    ## Flight Path Traces
    if 'routes' in airport_visibility:
        lats_plot, lons_plot = [] , []

        for row in route_df.itertuples(index = False):   

            lats, lons = zip(*row.Flight_Path)
            lats, lons = list(lats), list(lons)

            lats_plot.extend(lats + [None])
            lons_plot.extend(lons + [None])

        # Add trace
        trace = (go.Scattermap(
            lon= lons_plot,
            lat= lats_plot,
            mode='lines',
            line=dict(width=.5, color='red'),
            opacity = 0.2, 
            hoverinfo = 'skip', 
            name = "Flights",
            visible = True
        ))
        all_traces.append(trace)
    
    ## Airport Traces
    airport_types = airport_df['Type'].unique().tolist()

    # Bluescale Color Palette
    airport_plot_colors = {
    'heliport' :        "rgb(100,151,177)" ,
    'seaplane_base': 	"rgb(179,205,224)",
    'balloonport' : 	"rgb(179,205,224)",
    'small_airport' :  "rgb(0,91,150)"  ,
    'medium_airport' :	"rgb(3,57,108)",
    'large_airport':   "rgb(1,31,75)"
    }

    airport_proximity_dict = {
        "large_airport" : 55560,    # 30 nautical miles
        "medium_airport" : 9260,    # 5 nautical miles
        "small_airport" : 5556,     # 3 nautical miles
        "heliport":  2778,          # 1.5 nautical miles
        "seaplane_base" : 5556,     # 3 nautical miles
        "balloonport" : 5556        # 3 nautical miles
    }

    # Plot Trace
    if 'airports' in airport_visibility:
        for airport_type in airport_types:
            # Filter by airport type
            airport_df_filtered = airport_df.loc[airport_df['Type'] == airport_type]

            # Airport Marker 
            airport_marker = (go.Scattermap(
            lon = airport_df_filtered['Longitude_Deg'],
            lat = airport_df_filtered['Latitude_Deg'],
            hoverinfo = 'skip',
            customdata = np.stack([
                                airport_df_filtered.apply(airport_textbox, axis=1),
                                ], axis = 1
            ),
            hovertemplate = (
                "%{customdata[0]}<br><br>"
            ),
            text = airport_df_filtered.apply(lambda row: f"IATA Code: {row['Iata_Code']}<br>Name: {row['Name']}", axis=1),
            mode = 'markers',
            marker = dict(
                size = 2,
                color = airport_plot_colors[airport_type],
                opacity = 1
                ),
                name = airport_type.replace('_', ' ').title(),
                visible = True
            ))
            all_traces.append(airport_marker)

            # Airport Radius

            lats_plot, lons_plot = [] , []

            for airport in airport_df_filtered.itertuples():
                
                lats, lons = zip(*airport.Airport_Radius)
                lats, lons = list(lats), list(lons)

                lats_plot.extend(lats + [None])
                lons_plot.extend(lons + [None])
            
            airport_radii = (go.Scattermap(
            lon = lons_plot,
            lat = lats_plot,
            hoverinfo = 'skip',
            mode = 'lines',
            line = dict(
                width = 1,
                color = airport_plot_colors[airport_type],
                ),
                name = None,
                showlegend = False,
                visible = True
            ))
            all_traces.append(airport_radii)

    ## Create Plotly figure 
    fig = go.Figure(data = all_traces)

    fig.update_layout(
        title_text = '',
        map_style = 'carto-darkmatter-nolabels',
        showlegend = True,
        clickmode='event+select',
        hovermode = 'closest',
        legend_title_text = legend_key.replace('_', ' '),
    )


    # Return figure
    return fig


# Query
legend_arg = 'Visual_Evidence'
state = None
event_type = ['Plane_Crash', 'Flying_Object']
apparition_type = None
haunt_date_range = None
holiday = None

if not legend_arg:
    legend_arg = {'Event_Type': get_legend_items(hp_df, 'Event_Type')} 

legend_arg = {f'{legend_arg}': get_legend_items(hp_df, f'{legend_arg}')} 

additional_tags = ['Haunted_Places_Date']


filtered_hp_df = filter_hp_df(
    hp_df,
    state,
    event_type,
    apparition_type,
    haunt_date_range,
    holiday,
)
filtered_route_df = filter_route_df(filtered_hp_df, route_df, flight_intersections)
filtered_airport_df = filter_airport_df(filtered_hp_df, airport_df, flight_intersections, airport_intersections, airport_types = ['small_airport'])
    




hp_interactive_darkmatter(filtered_hp_df, filtered_route_df, filtered_airport_df, legend_arg, additional_tags, airport_visibility = ['airports'])

